In [1]:
# %pip install python-dotenv pandas clickhouse-connect
# !pip install tabulate

In [2]:
import sys
sys.path.append("..")
from src.db import *
from src.cleaning import *
from src.metrics import *
import pandas as pd
from IPython.display import Markdown, display
import re
import difflib
import numpy as np

# Giai đoạn 0 - Chuẩn bị kỹ thuật 

In [3]:
# 0.1 Xác nhận schema thực tế

df_schema = get_schema()
print(f"Số bảng: {df_schema['table'].nunique()}")
df_schema.head()

Số bảng: 16


,table,column_name,data_type
0,brand,changed_by_fk,Nullable(Int64)
1,brand,changed_on,Nullable(DateTime64(3))
2,brand,created_by_fk,Nullable(Int64)
3,brand,created_on,Nullable(DateTime64(3))
4,brand,elton_created_at,Nullable(DateTime64(3))


In [4]:
# 0.2 Vẽ ERD & phân loại nhóm bảng

mermaid_code = "erDiagram\n"
for table, group in df_schema.groupby("table"):
    mermaid_code += f"    {table} {{\n"
    for _, row in group.iterrows():
        clean_type = row["data_type"].replace(" ", "_").replace("(", "_").replace(")", "")
        clean_col = row['column_name'].replace('.', '_')
        mermaid_code += f"        {clean_type} {clean_col}\n"
    mermaid_code += "    }\n\n"

mermaid_code += """
    %% ===== Quan hệ chính dùng để JOIN khi phân tích (ClickHouse không có FK) =====
    laptop_model }|--|| brand : "thuoc ve"
    laptop_model }|--|| cpu_model : "su dung"
    laptop_model }|--|| gpu_model : "su dung"
    laptop_benchmark_result }|--|| laptop_model : "ket qua test cua"
    user_event_tracking }o--|| laptop_model : "hanh vi (qua JSON_EXTRACT device_id)"

    %% ===== Bang denormalized dung de QA/doi chieu, KHONG thay the viec tu JOIN =====
    int_dim_laptop ||--|| laptop_model : "QA - flat table doi chieu ket qua tu JOIN 5 bang"
    int_events_enriched }o--|| laptop_model : "QA - resolved_laptop_model_id vs toInt64(device_id)"
    int_comparison_chart_device }o--|| laptop_model : "QA - component click So sanh trong interest_score"
"""

display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

notes_md = """
## Ghi chú phân loại bảng

**Bảng chính dùng để JOIN khi phân tích (Bước 1 - sql/extraction.sql):**
`laptop_model`, `brand`, `cpu_model`, `gpu_model`, `laptop_benchmark_result`, `user_event_tracking`

**Bảng denormalized — dùng để QA/đối chiếu, KHÔNG thay thế việc tự viết JOIN:**
- `int_dim_laptop`: flat table có sẵn brand/cpu/gpu/benchmark theo từng laptop.
- `int_events_enriched`: có cả `laptop_model_id` (thô) và `resolved_laptop_model_id` (đã xử lý).
- `int_comparison_chart_device`: nguồn đối chiếu cho thành phần "click So sánh" trong `interest_score`.

**Bảng không có khóa liên kết — chỉ tham khảo, không JOIN:**
- `laptop_catalog`: quyết định chi tiết xem trong docs/project_notes.md (mục laptop_catalog).

**Lớp staging (dbt-style) — nguồn cho bảng chính, KHÔNG dùng trực tiếp:**
`stg_brand`, `stg_cpu_model`, `stg_gpu_model`, `stg_laptop_model`, `stg_laptop_benchmark_result`, `stg_user_event_tracking`
"""

with open("../docs/erd.md", "w", encoding="utf-8") as f:
    f.write(f"# ERD - laplaptech database\n\n```mermaid\n{mermaid_code}\n```\n{notes_md}")

print(sorted(df_schema['table'].unique()))

```mermaid
erDiagram
    brand {
        Nullable_Int64 changed_by_fk
        Nullable_DateTime64_3 changed_on
        Nullable_Int64 created_by_fk
        Nullable_DateTime64_3 created_on
        Nullable_DateTime64_3 elton_created_at
        Nullable_Int64 id
        Nullable_Bool is_chip_brand
        Nullable_String name
    }

    cpu_model {
        Nullable_Int64 brand_id
        Nullable_Int64 changed_by_fk
        Nullable_DateTime64_3 changed_on
        Nullable_Int64 created_by_fk
        Nullable_DateTime64_3 created_on
        Nullable_DateTime64_3 elton_created_at
        Nullable_Int64 id
        Nullable_Bool is_active
        Nullable_String name
    }

    gpu_model {
        Nullable_Int64 brand_id
        Nullable_Int64 changed_by_fk
        Nullable_DateTime64_3 changed_on
        Nullable_Int64 created_by_fk
        Nullable_DateTime64_3 created_on
        Nullable_DateTime64_3 elton_created_at
        Nullable_Int64 id
        Nullable_Bool is_active
        Nullable_String name
    }

    int_comparison_chart_device {
        Nullable_String brand_name
        Nullable_Int64 event_id
        Nullable_Int64 event_local_ts
        Nullable_Int64 laptop_model_id
        Nullable_String laptop_model_name
        Nullable_String session_id
        Nullable_String sort_by
        Nullable_String sort_direction
        Nullable_String user_psuedo_id
    }

    int_dim_laptop {
        Nullable_String brand_name
        Nullable_String cpu_name
        Nullable_Float64 gaming_battery_result_minutes
        Nullable_Float64 geekbench_6_compute_gpu_plugged_in
        Nullable_Float64 geekbench_6_cpu_multi_core_plugged_in
        Nullable_String gpu_name
        Nullable_Bool is_active
        Nullable_Bool is_gaming_laptop
        Nullable_Bool is_mobile_device
        Nullable_Bool is_workstation
        Nullable_Int64 l_laptop_model_id
        Nullable_String laptop_model_name
        Nullable_Float64 laptop_weight
        Nullable_Float64 office_battery_result_minutes
        Nullable_Float64 screen_size
        Nullable_Int64 year_introduce
    }

    int_events_enriched {
        Nullable_String app_version
        Nullable_String brand_name
        Nullable_String cpu_name
        Nullable_Int64 event_local_ts
        Nullable_String event_name
        Nullable_Int64 event_server_ts
        Nullable_UInt8 funnel_step
        Nullable_String gpu_name
        Nullable_Int64 id
        Nullable_Int64 l_laptop_model_id
        Nullable_String laptop_model_name
        Nullable_String page_name
        Nullable_Int64 page_number
        Nullable_String page_url
        Nullable_String referrer
        Nullable_Int64 resolved_laptop_model_id
        Nullable_String search_keyword
        Nullable_String session_id
        Nullable_Int64 user_id
        Nullable_String user_psuedo_id
    }

    laptop_benchmark_result {
        Nullable_Int64 changed_by_fk
        Nullable_DateTime64_3 changed_on
        Nullable_Int64 created_by_fk
        Nullable_DateTime64_3 created_on
        Nullable_DateTime64_3 elton_created_at
        Nullable_Float64 foldable_opening_battery_result_minutes
        Nullable_Float64 gaming_battery_result_minutes
        Nullable_Float64 geekbench_6_compute_gpu_battery
        Nullable_Float64 geekbench_6_compute_gpu_plugged_in
        Nullable_Float64 geekbench_6_cpu_multi_core_battery
        Nullable_Float64 geekbench_6_cpu_multi_core_plugged_in
        Nullable_Float64 geekbench_6_cpu_single_core_battery
        Nullable_Float64 geekbench_6_cpu_single_core_plugged_in
        Nullable_Float64 geekbench_7_compute_gpu_battery
        Nullable_Float64 geekbench_7_compute_gpu_plugged_in
        Nullable_Float64 geekbench_7_cpu_multi_core_battery
        Nullable_Float64 geekbench_7_cpu_multi_core_plugged_in
        Nullable_Float64 geekbench_7_cpu_single_core_battery
        Nullable_Float64 geekbench_7_cpu_single_core_plugged_in
        Nullable_Int64 id
        Nullable_Bool is_active
        Nullable_Int64 laptop_model_id
        Nullable_String note
        Nullable_Float64 office_battery_result_minutes
        Nullable_String review_video_url
    }

    laptop_catalog {
        Nullable_String brand
        Nullable_String cpu
        Nullable_String gpu
        Nullable_String name
    }

    laptop_model {
        Nullable_Float64 battery_capacity_whr
        Nullable_Int64 brand_id
        Nullable_String brand_model_codename
        Nullable_Int64 changed_by_fk
        Nullable_DateTime64_3 changed_on
        Nullable_Float64 charger_weight
        Nullable_Int64 cpu_model_id
        Nullable_String cpu_note
        Nullable_String cpu_tdp
        Nullable_Int64 created_by_fk
        Nullable_DateTime64_3 created_on
        Nullable_DateTime64_3 elton_created_at
        Nullable_Int64 gpu_model_id
        Nullable_String gpu_note
        Nullable_String gpu_tdp
        Nullable_Int64 id
        Nullable_Bool is_active
        Nullable_Bool is_gaming_laptop
        Nullable_Bool is_mobile_device
        Nullable_Bool is_visible
        Nullable_Bool is_workstation
        Nullable_Float64 laptop_weight
        Nullable_String name
        Nullable_Float64 screen_dimension_height
        Nullable_Float64 screen_dimension_width
        Nullable_Float64 screen_ppi
        Nullable_Float64 screen_size
        Nullable_String thumbnail_image_url
        Nullable_Int64 year_introduce
    }

    stg_brand {
        Nullable_Int64 brand_id
        Nullable_String brand_name
        Nullable_Bool is_chip_brand
    }

    stg_cpu_model {
        Nullable_Int64 brand_id
        Nullable_Int64 cpu_model_id
        Nullable_String cpu_name
        Nullable_Bool is_active
    }

    stg_gpu_model {
        Nullable_Int64 brand_id
        Nullable_Int64 gpu_model_id
        Nullable_String gpu_name
        Nullable_Bool is_active
    }

    stg_laptop_benchmark_result {
        Nullable_Float64 foldable_opening_battery_result_minutes
        Nullable_Float64 gaming_battery_result_minutes
        Nullable_Float64 geekbench_6_compute_gpu_battery
        Nullable_Float64 geekbench_6_compute_gpu_plugged_in
        Nullable_Float64 geekbench_6_cpu_multi_core_battery
        Nullable_Float64 geekbench_6_cpu_multi_core_plugged_in
        Nullable_Float64 geekbench_6_cpu_single_core_battery
        Nullable_Float64 geekbench_6_cpu_single_core_plugged_in
        Nullable_Bool is_active
        Nullable_Int64 laptop_benchmark_result_id
        Nullable_Int64 laptop_model_id
        Nullable_Float64 office_battery_result_minutes
    }

    stg_laptop_model {
        Nullable_Float64 battery_capacity_whr
        Nullable_Int64 brand_id
        Nullable_String brand_model_codename
        Nullable_Float64 charger_weight
        Nullable_Int64 cpu_model_id
        Nullable_String cpu_tdp
        Nullable_Int64 gpu_model_id
        Nullable_String gpu_tdp
        Nullable_Bool is_active
        Nullable_Bool is_gaming_laptop
        Nullable_Bool is_mobile_device
        Nullable_Bool is_visible
        Nullable_Bool is_workstation
        Nullable_Int64 laptop_model_id
        Nullable_String laptop_model_name
        Nullable_Float64 laptop_weight
        Nullable_Float64 screen_size
        Nullable_Int64 year_introduce
    }

    stg_user_event_tracking {
        Nullable_String app_version
        Nullable_String device
        Nullable_String event_data_raw
        Nullable_Int64 event_local_ts
        Nullable_String event_name
        Nullable_Int64 event_server_ts
        Nullable_Int64 id
        Nullable_String page_name
        Nullable_String page_url
        Nullable_String referrer
        Nullable_String session_id
        Nullable_Int64 user_id
        Nullable_String user_psuedo_id
    }

    user_event_tracking {
        Nullable_String app_version
        Nullable_String device
        Nullable_DateTime64_3 elton_created_at
        Nullable_String event_data
        Nullable_Int64 event_local_timestamp
        Nullable_String event_name
        Nullable_Int64 event_received_on_server_timestamp
        Nullable_Int64 id
        Nullable_String session_id
        Nullable_Int64 user_id
        Nullable_String user_psuedo_id
    }


    %% ===== Quan hệ chính dùng để JOIN khi phân tích (ClickHouse không có FK) =====
    laptop_model }|--|| brand : "thuoc ve"
    laptop_model }|--|| cpu_model : "su dung"
    laptop_model }|--|| gpu_model : "su dung"
    laptop_benchmark_result }|--|| laptop_model : "ket qua test cua"
    user_event_tracking }o--|| laptop_model : "hanh vi (qua JSON_EXTRACT device_id)"

    %% ===== Bang denormalized dung de QA/doi chieu, KHONG thay the viec tu JOIN =====
    int_dim_laptop ||--|| laptop_model : "QA - flat table doi chieu ket qua tu JOIN 5 bang"
    int_events_enriched }o--|| laptop_model : "QA - resolved_laptop_model_id vs toInt64(device_id)"
    int_comparison_chart_device }o--|| laptop_model : "QA - component click So sanh trong interest_score"

```

['brand', 'cpu_model', 'gpu_model', 'int_comparison_chart_device', 'int_dim_laptop', 'int_events_enriched', 'laptop_benchmark_result', 'laptop_catalog', 'laptop_model', 'stg_brand', 'stg_cpu_model', 'stg_gpu_model', 'stg_laptop_benchmark_result', 'stg_laptop_model', 'stg_user_event_tracking', 'user_event_tracking']


In [5]:
decision_note = """
## [Giai đoạn 0] Quyết định về cách xử lý các bảng int_* / stg_*

- Vẫn TỰ viết SQL JOIN 5 bảng (laptop_model, brand, cpu_model, gpu_model, laptop_benchmark_result)
  và TỰ viết JSON_EXTRACT thủ công từ user_event_tracking.event_data để luyện tập (Hướng A).
- int_dim_laptop / int_events_enriched / int_comparison_chart_device chỉ dùng làm nguồn đối chiếu
  (QA) sau khi tự làm xong, không dùng làm nguồn chính.
"""
append_note(decision_note)
print("Đã ghi quyết định Hướng A vào docs/project_notes.md")

Đã ghi quyết định Hướng A vào docs/project_notes.md


In [6]:
# 0.3 Kiểm tra khả năng dùng laptop_catalog

# Tính tỷ lệ match tên (exact + fuzzy)

def normalize_name(s: str) -> str:
    if pd.isna(s):
        return ""
    s = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", s.lower().strip())).strip()
    return s

df_catalog = query_cached("SELECT name FROM laplaptech.laptop_catalog", cache_dir="../data/raw")
df_model = query_cached("SELECT id, name FROM laplaptech.laptop_model", cache_dir="../data/raw")

df_catalog["name_norm"] = df_catalog["name"].apply(normalize_name)
df_model["name_norm"] = df_model["name"].apply(normalize_name)
model_names_norm = set(df_model["name_norm"])

df_catalog["exact_match"] = df_catalog["name_norm"].isin(model_names_norm)
unmatched = ~df_catalog["exact_match"]
df_catalog.loc[unmatched, "fuzzy_match"] = df_catalog.loc[unmatched, "name_norm"].apply(
    lambda x: (difflib.get_close_matches(x, model_names_norm, n=1, cutoff=0.9) or [None])[0]
)

exact_rate = df_catalog["exact_match"].mean()
fuzzy_rate = df_catalog.get("fuzzy_match", pd.Series(dtype=object)).notna().mean()
total_match_rate = exact_rate + fuzzy_rate

print(f"Tổng dòng: {len(df_catalog)} | Exact: {exact_rate:.1%} | Fuzzy bổ sung: {fuzzy_rate:.1%} | Tổng: {total_match_rate:.1%}")

Tổng dòng: 159 | Exact: 100.0% | Fuzzy bổ sung: 0.0% | Tổng: 100.0%


In [7]:
# Sanity-check độ tin cậy (collision & fanout)

dup_model_names = df_model.groupby("name_norm").filter(lambda g: len(g) > 1)

merged = df_catalog.merge(df_model[["id", "name", "name_norm"]], on="name_norm", how="left", suffixes=("_catalog", "_model"))
match_counts = merged.groupby("name_catalog")["id"].nunique()
multi_match = match_counts[match_counts > 1]

is_subset = len(df_catalog) <= len(df_model)
reliable = len(dup_model_names) == 0 and len(multi_match) == 0

print(f"laptop_model trùng name_norm: {len(dup_model_names)} dòng")
print(f"laptop_catalog match >1 id:  {len(multi_match)} dòng")
print(f"laptop_catalog là tập con của laptop_model: {is_subset}")
if len(dup_model_names) > 0:
    display(dup_model_names.sort_values("name_norm"))

laptop_model trùng name_norm: 17 dòng
laptop_catalog match >1 id:  8 dòng
laptop_catalog là tập con của laptop_model: True


,id,name,name_norm
10,11,Acer Nitro V 15,acer nitro v 15
107,108,Acer Nitro V 15,acer nitro v 15
8,9,Asus Vivobook S 14,asus vivobook s 14
82,83,Asus Vivobook S 14,asus vivobook s 14
116,117,ASUS Vivobook S14,asus vivobook s14
115,116,ASUS Vivobook S14,asus vivobook s14
104,105,ASUS Vivobook S14,asus vivobook s14
152,153,Asus Zenbook A14,asus zenbook a14
58,59,Asus Zenbook A14,asus zenbook a14
133,134,Asus Zenbook S 16,asus zenbook s 16


In [8]:
# Quyết định cuối cùng & flag cho Giai đoạn 2

final_decision = f"""
## [Giai đoạn 0] Quyết định CUỐI CÙNG về laptop_catalog

Match tên: {total_match_rate:.1%} ({len(df_catalog)} dòng, exact: {exact_rate:.1%}, fuzzy: {fuzzy_rate:.1%}).
Sanity-check: {len(dup_model_names)} dòng laptop_model bị trùng name_norm,
{len(multi_match)} dòng laptop_catalog match >1 id, là tập con của laptop_model: {is_subset}.

QUYẾT ĐỊNH: {"match đáng tin cậy nhưng laptop_catalog dư thừa (không có ID, thông tin đã có sẵn chuẩn hơn ở laptop_model+brand+cpu_model+gpu_model)" if reliable else "phát hiện collision/fanout → 100% match ban đầu KHÔNG đáng tin cậy"}.
→ Loại laptop_catalog khỏi phạm vi phân tích chính thức (sql/extraction.sql), không dùng để JOIN dưới bất kỳ hình thức nào.
"""

append_note(final_decision)

dq_flag = f"""
## [Flag cho Giai đoạn 2 - Data Quality Audit] laptop_model có tên trùng lặp

{len(dup_model_names)} dòng trong laptop_model có name_norm trùng dòng khác (id khác nhau,
không ảnh hưởng JOIN qua id, nhưng cần rà lại khi viết báo cáo/content theo tên ở Giai đoạn 4).
Id liên quan: {sorted(dup_model_names['id'].tolist())}
"""
append_note(dq_flag)

print(final_decision)


## [Giai đoạn 0] Quyết định CUỐI CÙNG về laptop_catalog

Match tên: 100.0% (159 dòng, exact: 100.0%, fuzzy: 0.0%).
Sanity-check: 17 dòng laptop_model bị trùng name_norm,
8 dòng laptop_catalog match >1 id, là tập con của laptop_model: True.

QUYẾT ĐỊNH: phát hiện collision/fanout → 100% match ban đầu KHÔNG đáng tin cậy.
→ Loại laptop_catalog khỏi phạm vi phân tích chính thức (sql/extraction.sql), không dùng để JOIN dưới bất kỳ hình thức nào.



# Giai đoạn 1 - Bối cảnh kinh doanh 

In [9]:
# 1.1 Bối cảnh kinh doanh theo từng đơn vị

df_context = pd.DataFrame([
    ["Đội ngũ Sản phẩm (Product/UX)", "Người dùng quan tâm chỉ số nào nhất (pin hay điểm Geekbench)? Tính năng nào thu hút nhất?"],
    ["Đội ngũ Nội dung/Review", "Mẫu laptop/CPU/GPU nào đang hot? Có 'điểm số thất vọng' nào làm chủ đề content không?"],
    ["Đội ngũ Kinh doanh/Affiliate", "Trade-off nào người dùng chấp nhận? Mẫu nào hiệu năng/giá trị tốt nhất để đẩy mạnh?"],
], columns=["Bối cảnh / Đơn vị", "Câu hỏi kinh doanh cần giải quyết"])

md = "## [Giai đoạn 1] 1.1 Bối cảnh kinh doanh\n\n" + df_context.to_markdown(index=False) + "\n"
append_note(md)

In [10]:
# 1.2 Biến trọng yếu

key_vars = pd.DataFrame([
    ["I. Hiệu năng & Năng lượng", "laptop_benchmark_result", "geekbench_6_cpu_multi_core_plugged_in, geekbench_6_cpu_multi_core_battery", "Đo mức sụt giảm hiệu năng khi dùng pin"],
    ["I. Hiệu năng & Năng lượng", "laptop_benchmark_result", "office_battery_result_minutes, gaming_battery_result_minutes", "Thời lượng pin theo tác vụ"],
    ["I. Hiệu năng & Năng lượng", "laptop_model", "battery_capacity_whr", "Dung lượng pin danh định"],
    ["II. Thiết kế & Di động", "laptop_model", "laptop_weight, charger_weight", "Tổng trọng lượng di chuyển"],
    ["II. Thiết kế & Di động", "laptop_model", "cpu_tdp, gpu_tdp", "Công suất tản nhiệt thiết kế"],
    ["III. Phân khúc & Định danh", "laptop_model", "is_gaming_laptop, is_workstation", "Phân loại phân khúc"],
    ["III. Phân khúc & Định danh", "laptop_model", "year_introduce", "Vòng đời sản phẩm"],
    ["III. Phân khúc & Định danh", "brand, cpu_model, gpu_model", "name", "Tên thương hiệu/chip để gom nhóm"],
    ["IV. Hành vi & Nhu cầu", "user_event_tracking", "event_name, event_data", "Nhận diện hành động/nội dung quan tâm"],
    ["IV. Hành vi & Nhu cầu", "user_event_tracking", "session_id, user_psuedo_id", "Hành trình người dùng"],
    ["IV. Hành vi & Nhu cầu", "laptop_benchmark_result", "review_video_url", "Điểm chạm nội dung"],
], columns=["Nhóm", "Bảng", "Biến", "Vai trò"])

# Đối chiếu từng cột với schema thật đã lấy ở Giai đoạn 0
real_cols = set(df_schema["table"] + "." + df_schema["column_name"])
def check_cols(row):
    table, cols = row["Bảng"], row["Biến"]
    missing = []
    for t in table.split(", "):
        for c in cols.replace(" ", "").split(","):
            if f"{t}.{c}" not in real_cols:
                missing.append(f"{t}.{c}")
    return ", ".join(missing) if missing else "OK"

key_vars["Đối chiếu schema thật"] = key_vars.apply(check_cols, axis=1)
mismatch = key_vars[key_vars["Đối chiếu schema thật"] != "OK"]
print(f"Số dòng lệch schema thật: {len(mismatch)}")
if len(mismatch) > 0:
    display(mismatch)

md = "## [Giai đoạn 1] 1.2 Biến trọng yếu (đã đối chiếu schema thật)\n\n" + key_vars.to_markdown(index=False) + "\n"
append_note(md)

Số dòng lệch schema thật: 0


In [11]:
# 1.3 Hypotheses

hypotheses = pd.DataFrame([
    ["H1", "Trục 1", "Hãng CPU/GPU giữ hiệu năng khi dùng pin tốt hơn hẳn hãng khác (CPU_Battery_Ratio)", "ANOVA/Kruskal-Wallis theo nhóm chip"],
    ["H2", "Trục 1", "TDP cao → CPU_Battery_Ratio thấp hơn đáng kể", "Spearman correlation"],
    ["H3", "Trục 1", "Pin/Whr khác biệt có ý nghĩa giữa các hãng CPU", "ANOVA/Kruskal-Wallis"],
    ["H4", "Trục 1", "Tồn tại nhóm 'Golden Ratio': top 25% pin/Whr đồng thời top 25% nhẹ", "Descriptive, percentile 2 chiều"],
    ["H5", "Trục 2", "Laptop gaming nhận interest_score cao hơn laptop văn phòng", "Mann-Whitney"],
    ["H6", "Trục 2", "Session xem review_video_url có session dài hơn", "Mann-Whitney"],
    ["H7", "Trục 2", "Mobile có tỷ lệ bounce cao hơn desktop", "Chi-square"],
    ["H8", "Trục 3", "Pin >8h nhận interest_score cao hơn gaming top 25% Geekbench", "Mann-Whitney, sensitivity 420/480/540"],
    ["H9", "Trục 3", "Tồn tại nhóm 'click nhiều nhưng benchmark kém'", "Descriptive, liệt kê danh sách"],
    ["H10", "Trục 3", "Gaming pin tốt (>300') nhận interest_score cao hơn gaming pin kém", "Mann-Whitney trong cùng phân khúc gaming"],
], columns=["ID", "Trục", "Hypothesis", "Kiểm định"])

note = """
**Điểm sửa quan trọng đã áp dụng:**
- H1/H3: group theo "hãng CPU" phải JOIN cpu_model.brand_id → brand.id, lọc is_chip_brand = true.
- interest_score (dùng cho H5, H8, H9, H10) = 1×pageview DeviceDetail + 2×click "So sánh" + 3×click review_video_url,
  tính theo laptop_model_id. Đối chiếu thành phần "click So sánh" với int_comparison_chart_device.
"""

md = "## [Giai đoạn 1] 1.3 Hypotheses\n\n" + hypotheses.to_markdown(index=False) + "\n" + note
append_note(md)

In [12]:
# 1.4 Prioritization

priority = pd.DataFrame([
    ["H1", 4, 2, 4, 8.0], ["H2", 3, 2, 4, 6.0], ["H3", 4, 2, 3, 6.0], ["H4", 3, 3, 4, 4.0],
    ["H5", 5, 3, 3, 5.0], ["H6", 3, 3, 3, 3.0], ["H7", 2, 3, 3, 2.0], ["H8", 5, 4, 3, 3.75],
    ["H9", 5, 3, 3, 5.0], ["H10", 4, 3, 3, 4.0],
], columns=["ID", "Impact", "Effort", "Data Confidence", "Priority Score"])

decision = """
**Quyết định: bộ 3 insight chính = H1, H5, H9** (thay vì H1, H9, H10).

Lý do: H1+H9+H10 khiến Trục 2 (User Engagement) không có đại diện nào, trong khi dashboard
(Trang 3 - User Funnel) và cấu trúc 3-trục của cả kế hoạch cần đủ 3 trục. H5 dùng chung
interest_score với H9/H10 nên không tốn thêm effort đáng kể. H10 giữ trong backlog, làm nếu còn thời gian.

Backlog (làm nếu còn thời gian): H2, H3, H4, H6, H7, H8, H10.

*(Lưu ý: quyết định này dựa trên Data Confidence ước tính tại Giai đoạn 1. Đã được xác minh
lại và điều chỉnh ở Giai đoạn 2 — xem mục "[Giai đoạn 2] Chốt population sạch & H1'" bên dưới,
trong đó H1 được thay bằng H1' do laptop_benchmark_result không đủ dữ liệu.)*
"""

md = "## [Giai đoạn 1] 1.4 Prioritization\n\n" + priority.to_markdown(index=False) + "\n" + decision
append_note(md)

In [13]:
# 1.5 Chuẩn hóa định dạng project_notes.md
path = "../docs/project_notes.md"
with open(path, "r", encoding="utf-8") as f:
    content = f.read()

import re
# Đảm bảo luôn có đúng 2 dòng trống trước mỗi "## " (trừ khi đang ở đầu file)
fixed = re.sub(r"\n*(##[^\n]*\n)", r"\n\n\1", content)
fixed = re.sub(r"\n{3,}", "\n\n", fixed).lstrip("\n")

with open(path, "w", encoding="utf-8") as f:
    f.write(fixed)

print("Đã chuẩn hóa dòng trống trong project_notes.md")

Đã chuẩn hóa dòng trống trong project_notes.md


# Giai đoạn 2 - Khám phá & Làm sạch dữ liệu

In [14]:
# Rà lại laptop_model bị trùng tên, phân loại re-release vs lỗi nhập liệu

df_dup_detail = query_cached(
    f"""SELECT id, name, year_introduce, brand_id, cpu_model_id
        FROM laplaptech.laptop_model
        WHERE id IN ({",".join(map(str, sorted(dup_model_names['id'].tolist())))})
        ORDER BY name, year_introduce""",
    cache_dir="../data/raw"
)
display(df_dup_detail)

# Heuristic: cùng tên nhưng year_introduce khác nhau -> nhiều khả năng re-release
# cùng tên VÀ cùng year_introduce -> nhiều khả năng lỗi nhập liệu (trùng dòng)
dup_summary = df_dup_detail.groupby("name")["year_introduce"].nunique()
likely_rerelease = dup_summary[dup_summary > 1].index.tolist()
likely_data_entry_error = dup_summary[dup_summary == 1].index.tolist()

note = f"""
## [Giai đoạn 2 - 2.0] Kết luận về laptop_model trùng tên

- Nhiều khả năng RE-RELEASE (cùng tên, khác year_introduce): {likely_rerelease}
- Nhiều khả năng LỖI NHẬP LIỆU (cùng tên, cùng year_introduce): {likely_data_entry_error}
- Quyết định: giữ nguyên tất cả các id này trong phân tích (không xóa), vì id vẫn là khóa duy nhất
  hợp lệ để JOIN; chỉ cẩn trọng khi viết báo cáo/content theo TÊN ở Giai đoạn 4.
"""
append_note(note)

,id,name,year_introduce,brand_id,cpu_model_id
0,105,ASUS Vivobook S14,2025,1,56
1,116,ASUS Vivobook S14,2025,1,42
2,117,ASUS Vivobook S14,2025,1,45
3,11,Acer Nitro V 15,2024,2,21
4,108,Acer Nitro V 15,2025,2,62
5,9,Asus Vivobook S 14,2024,1,1
6,83,Asus Vivobook S 14,2025,1,56
7,97,Asus ZenBook Test device,2024,1,1
8,98,Asus ZenBook Test device,2024,1,1
9,59,Asus Zenbook A14,2025,1,42


### 2.1 Data Profiling có số liệu

In [15]:
df_benchmark = query_cached(
    "SELECT * FROM laplaptech.laptop_benchmark_result",
    cache_dir="../data/raw", force_refresh=False
)

df_laptop = query_cached("SELECT * FROM laplaptech.laptop_model", cache_dir="../data/raw")

benchmark_cols = [
    "geekbench_6_cpu_multi_core_plugged_in", "geekbench_6_cpu_multi_core_battery",
    "office_battery_result_minutes", "gaming_battery_result_minutes",
]
laptop_cols = ["battery_capacity_whr", "laptop_weight", "charger_weight"]

profile_benchmark = profile_dataframe(df_benchmark, benchmark_cols)
profile_laptop = profile_dataframe(df_laptop, laptop_cols)

display(profile_benchmark)
display(profile_laptop)

md = ("## [Giai đoạn 2 - 2.1] Data Profiling\n\n### laptop_benchmark_result\n\n"
      + profile_benchmark.to_markdown(index=False) + "\n\n### laptop_model\n\n"
      + profile_laptop.to_markdown(index=False) + "\n")
append_note(md)

,column,null_pct,min,max,outlier_count,skew
0,geekbench_6_cpu_multi_core_plugged_in,0.4,7949.0,22252.0,0.0,0.617
1,geekbench_6_cpu_multi_core_battery,0.0,6895.0,21259.0,1.0,0.723
2,office_battery_result_minutes,0.0,525.0,706.0,1.0,-0.894
3,gaming_battery_result_minutes,1.0,NaN,NaN,NaN,NaN


,column,null_pct,min,max,outlier_count,skew
0,battery_capacity_whr,0.0314,0.0,121.3,16,-0.726
1,laptop_weight,0.0314,0.0,3620.0,9,0.002
2,charger_weight,0.3962,0.0,1199.0,0,0.874


### 2.2 Xử lý vấn đề kỹ thuật

In [16]:
# Parse TDP từ String sang số

df_laptop["cpu_tdp_parsed"] = parse_tdp_column(df_laptop, "cpu_tdp")
df_laptop["gpu_tdp_parsed"] = parse_tdp_column(df_laptop, "gpu_tdp")

print(f"Số dòng parse thất bại (NaN) - cpu_tdp: {df_laptop['cpu_tdp_parsed'].isna().sum()}")
print(f"Số dòng parse thất bại (NaN) - gpu_tdp: {df_laptop['gpu_tdp_parsed'].isna().sum()}")
display(df_laptop[["cpu_tdp", "cpu_tdp_parsed", "gpu_tdp", "gpu_tdp_parsed"]].head(10))

note = """
## [Giai đoạn 2 - 2.2.1] Parse cpu_tdp/gpu_tdp

Quyết định: với TDP dạng dải (VD "15-28W"), lấy TRUNG BÌNH của dải làm giá trị đại diện
(không lấy min hoặc max riêng lẻ) vì đây là ước lượng cân bằng, phù hợp cho correlation ở H2.
Logic parse: hàm parse_tdp() trong src/cleaning.py.
"""
append_note(note)

Số dòng parse thất bại (NaN) - cpu_tdp: 141
Số dòng parse thất bại (NaN) - gpu_tdp: 125


,cpu_tdp,cpu_tdp_parsed,gpu_tdp,gpu_tdp_parsed
0,28,28.0,NaN,NaN
1,Tiêu thụ trung bình 31W,31.0,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,NaN,NaN,100W,100.0
5,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN


In [17]:
# Đọc query tỷ lệ NULL Geekbench 6/7 theo year_introduce (từ sql/exploration.sql)

gb_query = """
SELECT
    m.year_introduce,
    COUNT(*) AS total_laptops,
    SUM(b.geekbench_6_cpu_multi_core_plugged_in IS NULL) AS null_gb6,
    SUM(b.geekbench_7_cpu_multi_core_plugged_in IS NULL) AS null_gb7
FROM laplaptech.laptop_benchmark_result b
INNER JOIN laplaptech.laptop_model m ON b.laptop_model_id = m.id
GROUP BY m.year_introduce
ORDER BY m.year_introduce
"""  # Đồng bộ với sql/exploration.sql
df_gb = query_cached(gb_query, cache_dir="../data/raw")
display(df_gb)

decision_gb = "COALESCE" if (df_gb["null_gb6"].sum() > 0 and df_gb["null_gb7"].sum() > 0) else "chỉ dùng GB6"

note = f"""
## [Giai đoạn 2 - 2.2.2] Quyết định Geekbench 6 vs 7

{df_gb.to_markdown(index=False)}

Quyết định: {decision_gb}. Toàn bộ hypothesis liên quan (H1, H2) chỉ dùng geekbench_6_*
theo phạm vi đã chốt ở Giai đoạn 1, KHÔNG mở rộng sang geekbench_7 trong bản đầu (giữ trong backlog nếu cần).
"""
append_note(note)

,year_introduce,total_laptops,null_gb6,null_gb7
0,2026,3,2,3


### 2.3 Data Quality Audit — ngưỡng cụ thể

In [18]:
# Gắn cờ outlier vật lý (pin > 1440 phút)

anomaly_office = flag_physical_outliers(df_benchmark, "office_battery_result_minutes", 1440)
anomaly_gaming = flag_physical_outliers(df_benchmark, "gaming_battery_result_minutes", 1440)
df_anomaly = pd.concat([anomaly_office, anomaly_gaming], ignore_index=True)

print(f"Số dòng bị gắn cờ anomaly (pin > 24h): {len(df_anomaly)}")
display(df_anomaly[["laptop_model_id", "anomaly_type", "anomaly_value", "anomaly_threshold"]])

df_anomaly.to_csv("../data/processed/anomaly_battery_minutes.csv", index=False)

Số dòng bị gắn cờ anomaly (pin > 24h): 0


,laptop_model_id,anomaly_type,anomaly_value,anomaly_threshold


In [19]:
# Phân loại lý do NULL cho gaming_battery_result_minutes

df_merged_bm = df_benchmark.merge(
    df_laptop[["id", "is_gaming_laptop", "is_workstation"]],
    left_on="laptop_model_id", right_on="id", how="left"
)
df_merged_bm["null_reason_gaming_battery"] = classify_null_reason(
    df_merged_bm, "gaming_battery_result_minutes"
)
print(df_merged_bm["null_reason_gaming_battery"].value_counts())

null_reason_gaming_battery
unclear    5
Name: count, dtype: int64


In [20]:
# Validate range CPU_Battery_Ratio

df_benchmark["cpu_battery_ratio"] = compute_cpu_battery_ratio(df_benchmark)

invalid_ratio = df_benchmark[
    (df_benchmark["cpu_battery_ratio"] <= 0) | (df_benchmark["cpu_battery_ratio"] > 1)
]
print(f"Số dòng CPU_Battery_Ratio bất thường (ngoài (0,1]): {len(invalid_ratio)}")
if len(invalid_ratio) > 0:
    display(invalid_ratio[["laptop_model_id", "cpu_battery_ratio"]])
    invalid_ratio.to_csv("../data/processed/anomaly_cpu_battery_ratio.csv", index=False)

Số dòng CPU_Battery_Ratio bất thường (ngoài (0,1]): 0


In [21]:
# Audit population, dùng logic phát hiện (tự thích ứng dữ liệu mới)

CONTAMINATED_DEVICE_IDS = detect_contaminated_devices(df_laptop)
print(f"Tổng số thiết bị bị loại khỏi population: {len(CONTAMINATED_DEVICE_IDS)}")

df_laptop_final = filter_non_laptop_devices(df_laptop)
df_laptop_final["laptop_weight_g"] = normalize_laptop_weight(df_laptop_final)
print(f"Population sạch: {len(df_laptop_final)} laptop thật (tổng gốc: {len(df_laptop)})")

# So sánh với lần audit trước (Giai đoạn 2, snapshot 157 dòng) để phát hiện dữ liệu mới
PREVIOUS_SNAPSHOT_IDS = {77, 78, 79, 81, 95, 96, 97, 98, 99, 100, 123, 125, 126, 127, 128, 129, 132, 137, 139, 141, 149, 156}
new_contaminated = set(CONTAMINATED_DEVICE_IDS) - PREVIOUS_SNAPSHOT_IDS
resolved_contaminated = PREVIOUS_SNAPSHOT_IDS - set(CONTAMINATED_DEVICE_IDS)
print(f"ID nhiễm MỚI phát hiện (chưa từng thấy ở snapshot trước): {new_contaminated}")
print(f"ID từng bị coi là nhiễm nhưng giờ KHÔNG còn (cần rà lại thủ công nếu có): {resolved_contaminated}")

note = f"""
## [Giai đoạn 2 - 2.3.4] Audit population (snapshot hiện tại: laptop_model = {len(df_laptop)} dòng)

Phát hiện {len(CONTAMINATED_DEVICE_IDS)} thiết bị KHÔNG phải laptop (logic phát hiện tự động,
So với snapshot Giai đoạn 2 gốc (159 dòng, 22 ID nhiễm):
- ID nhiễm MỚI: {sorted(new_contaminated) if new_contaminated else "Không có"}
- ID không còn bị coi là nhiễm: {sorted(resolved_contaminated) if resolved_contaminated else "Không có"}
Population sạch hiện tại: {len(df_laptop_final)}/{len(df_laptop)} dòng.
"""
append_note(note)
print(note)

Tổng số thiết bị bị loại khỏi population: 26
Population sạch: 133 laptop thật (tổng gốc: 159)
ID nhiễm MỚI phát hiện (chưa từng thấy ở snapshot trước): {159, 80, 158, 135}
ID từng bị coi là nhiễm nhưng giờ KHÔNG còn (cần rà lại thủ công nếu có): set()

## [Giai đoạn 2 - 2.3.4] Audit population (snapshot hiện tại: laptop_model = 159 dòng)

Phát hiện 26 thiết bị KHÔNG phải laptop (logic phát hiện tự động,
So với snapshot Giai đoạn 2 gốc (159 dòng, 22 ID nhiễm):
- ID nhiễm MỚI: [80, 135, 158, 159]
- ID không còn bị coi là nhiễm: Không có
Population sạch hiện tại: 133/159 dòng.



### 2.4 Tính các chỉ số phái sinh, lưu ra data/processed/

In [22]:
# Tính chỉ số phái sinh TRÊN POPULATION ĐÃ SẠCH 
df_laptop_metrics = df_laptop_final.merge(
    df_benchmark[["laptop_model_id", "cpu_battery_ratio", "office_battery_result_minutes"]],
    left_on="id", right_on="laptop_model_id", how="left"
)
# battery_density tính trên laptop_weight_g đã chuẩn hóa, không dùng laptop_weight gốc
df_laptop_metrics["battery_density"] = (
    pd.to_numeric(df_laptop_metrics["battery_capacity_whr"], errors="coerce")
    / df_laptop_metrics["laptop_weight_g"].replace(0, np.nan)
)
df_laptop_metrics["office_battery_per_whr"] = compute_office_battery_per_whr(df_laptop_metrics)

df_laptop_metrics = select_report_columns(df_laptop_metrics)
df_laptop_metrics.to_csv("../data/processed/laptop_metrics.csv", index=False)

display(df_laptop_metrics.groupby("is_gaming_laptop")["battery_density"].agg(["count", "mean", "std"]))

[select_report_columns] Cảnh báo: các cột sau không có trong df, đã bỏ qua: {'compare_count', 'pageview_count', 'interest_score'}


,count,mean,std
is_gaming_laptop,,,
False,80,0.048606,0.010439
True,47,0.037152,0.008991


In [23]:
# Kiểm tra key JSON có đồng nhất giữa các app_version không

df_events_raw = query_cached(
    "SELECT event_name, app_version, event_data FROM laplaptech.user_event_tracking LIMIT 50000",
    cache_dir="../data/raw"
)

df_inventory = inventory_json_keys(df_events_raw, json_col="event_data")
display(df_inventory)

df_issues = find_key_inconsistencies(df_inventory)
print(f"Số event_name có key KHÔNG đồng nhất giữa các version: {len(df_issues)}")
if len(df_issues) > 0:
    display(df_issues)

note = f"""
## [Giai đoạn 2 - 2.4b] Kiểm tra đồng nhất JSON key theo app_version

Số event_name có key lệch giữa các version: {len(df_issues)}
{df_issues.to_markdown(index=False) if len(df_issues) > 0 else "Không phát hiện lệch key — an toàn để JSON_EXTRACT trực tiếp ở Bước 2.5.1/sql/extraction.sql."}
"""
append_note(note)

,event_name,app_version,keys_found,n_rows
0,comparison_chart_sort_selection,web,"[device_ids, sort_by, sort_direction]",435
1,load_more_device_home,web,[page],12433
2,pageview,web,"[device_id, page_name, referrer, url]",19936
3,search_for_device,web,[keyword],12792
4,select_device_for_comparison,web,[device_id],4404


Số event_name có key KHÔNG đồng nhất giữa các version: 0


### 2.5 Test JSON_EXTRACT trên user_event_tracking (query đồng bộ sql/exploration.sql)

In [24]:
event_query = """
SELECT
    id AS event_id, session_id, event_name,
    toInt64OrNull(JSONExtractString(event_data, 'device_id')) AS laptop_id_extracted,
    event_local_timestamp
FROM laplaptech.user_event_tracking
WHERE event_name = 'pageview'
  AND JSONExtractString(event_data, 'page_name') = 'DeviceDetail'
ORDER BY id
LIMIT 1000
"""
df_events_test = query_cached(event_query, cache_dir="../data/raw", force_refresh=False)

orphan_rate = df_events_test["laptop_id_extracted"].isna().mean()
print(f"Tỷ lệ device_id không parse được thành số (orphan tạm thời): {orphan_rate:.1%}")
display(df_events_test.drop(columns=["session_id"]).head())

Tỷ lệ device_id không parse được thành số (orphan tạm thời): 0.0%


,event_id,event_name,laptop_id_extracted,event_local_timestamp
0,1,pageview,55,1736877740
1,3,pageview,54,1736877768
2,5,pageview,49,1736877770
3,7,pageview,47,1736877774
4,11,pageview,55,1736879683


In [25]:
# So sánh laptop_id tự extract với resolved_laptop_model_id (bảng QA)

min_id, max_id = df_events_test["event_id"].min(), df_events_test["event_id"].max()
df_enriched = query_cached(
    f"""SELECT id, `l.laptop_model_id` AS laptop_model_id, resolved_laptop_model_id
        FROM laplaptech.int_events_enriched
        WHERE id BETWEEN {min_id} AND {max_id}""",
    cache_dir="../data/raw", force_refresh=False
)

compare = df_events_test.merge(df_enriched, left_on="event_id", right_on="id", how="inner", suffixes=("", "_enriched"))
print(f"Số dòng đối chiếu được: {len(compare)}")  # phải > 0 mới tính mismatch_rate có ý nghĩa
mismatch = compare[compare["laptop_id_extracted"] != compare["resolved_laptop_model_id"]]
mismatch_rate = len(mismatch) / len(compare) if len(compare) > 0 else None
print(f"Tỷ lệ mismatch: {mismatch_rate}")
note = f"""
## [Giai đoạn 2 - 2.5.2] Đối chiếu JSON_EXTRACT thủ công với int_events_enriched

Tỷ lệ mismatch: {mismatch_rate:.1%} trên {len(compare)} dòng đối chiếu.
Quyết định: {"chấp nhận được, tiếp tục dùng cách tự extract cho sql/extraction.sql" if mismatch_rate < 0.05 else "tỷ lệ lệch cao, cần rà lại logic resolve trước khi viết extraction.sql chính thức"}.
"""
append_note(note)


Số dòng đối chiếu được: 1000
Tỷ lệ mismatch: 0.0


In [26]:
# Dựng hành trình người dùng theo session
df_journey = build_session_journey(df_events_test, session_col="session_id", ts_col="event_local_timestamp")
display(df_journey.drop(columns=["session_id"], errors="ignore").head(15))

,event_id,event_name,laptop_id_extracted,event_local_timestamp,step_in_session
314,1496,pageview,35,1737045489,1
510,2891,pageview,35,1737069894,1
625,3649,pageview,55,1737076271,1
842,5133,pageview,50,1737092970,1
845,5139,pageview,44,1737093009,2
899,5542,pageview,3,1737098374,1
299,1359,pageview,46,1737044762,1
300,1363,pageview,46,1737044808,2
674,3987,pageview,55,1737079889,1
676,3992,pageview,54,1737079938,2


### 2.6 Test thủ công các hàm trong src/cleaning.py và src/metrics.py

In [27]:
# parse_tdp
assert parse_tdp("45W") == 45.0
assert parse_tdp("15-28W") == 21.5
assert pd.isna(parse_tdp(None))
assert pd.isna(parse_tdp(""))

# compute_cpu_battery_ratio — tránh chia cho 0
test_df = pd.DataFrame({
    "geekbench_6_cpu_multi_core_battery": [800, 500, 300],
    "geekbench_6_cpu_multi_core_plugged_in": [1000, 0, None],
})
result = compute_cpu_battery_ratio(test_df)
assert result.iloc[0] == 0.8
assert pd.isna(result.iloc[1])  # chia cho 0 -> NaN, không lỗi
assert pd.isna(result.iloc[2])  # NULL plugged_in -> NaN

print("Tất cả unit test PASS")

Tất cả unit test PASS


In [28]:
summary = f"""
## [Giai đoạn 2] Tổng kết

Đã hoàn thành: profiling, parse TDP, quyết định Geekbench 6/7, Data Quality Audit đầy đủ
(outlier vật lý, rule NULL, validate CPU_Battery_Ratio, audit population — loại
{len(CONTAMINATED_DEVICE_IDS)} thiết bị nhiễm + chuẩn hóa đơn vị laptop_weight), kiểm tra đồng nhất
JSON key, xử lý event tracking (JSON_EXTRACT, đối chiếu QA, session journey), unit test.

Phát hiện quan trọng: laptop_benchmark_result chỉ có 3/157 dòng -> H1 gốc không khả thi,
thay bằng H1' (trade-off trọng lượng/pin, dùng laptop_model, không phụ thuộc benchmark).

Output chính thức DUY NHẤT: data/processed/laptop_metrics.csv (đã loại contamination, đã chuẩn
hóa đơn vị). Sẵn sàng chuyển sang Bước 1: viết sql/extraction.sql.
"""
append_note(summary)

# Giai đoạn 3 - Khám phá Insights chuyên sâu

### Điều tra event thật trước khi build interest_score

Trước khi build interest_score, cần xác nhận: 
(1) có event/trang nào cho "xem video review" không
(2) event nào đại diện đúng cho hành động "So sánh". Chi tiết đầy đủ từng bước điều tra đã được ghi vào docs/project_notes.md; ở đây chỉ giữ lại các bước dẫn tới kết luận cuối cùng.

In [29]:
df_event_names = query_cached(
    "SELECT event_name, COUNT(*) AS total_events, COUNT(DISTINCT session_id) AS distinct_sessions "
    "FROM laplaptech.user_event_tracking GROUP BY event_name ORDER BY total_events DESC",
    cache_dir="../data/raw", force_refresh=False
)
display(df_event_names)

df_page_names = query_cached(
    "SELECT JSONExtractString(event_data, 'page_name') AS page_name, COUNT(*) AS total_events "
    "FROM laplaptech.user_event_tracking WHERE event_name = 'pageview' "
    "GROUP BY page_name ORDER BY total_events DESC",
    cache_dir="../data/raw", force_refresh=False
)
display(df_page_names)

append_note(
    "## [Giai đoạn 3 - 0] event_name và page_name thực tế\n\n"
    "### event_name\n\n" + df_event_names.to_markdown(index=False) + "\n\n"
    "### page_name (trong pageview)\n\n" + df_page_names.to_markdown(index=False) + "\n\n"
    "**Kết luận:** không có event_name hoặc page_name nào tương ứng với 'xem video review' "
    "-> thành phần này không có dữ liệu tracking, sẽ bị loại khỏi interest_score.\n"
)

,event_name,total_events,distinct_sessions
0,search_for_device,302119,37884
1,pageview,258948,72301
2,load_more_device_home,180502,56178
3,select_device_for_comparison,29579,8557
4,add_to_comparison,6298,4740
5,comparison_chart_sort_selection,3519,1339
6,user_login,590,289
7,comparison_chart_geekbench_version_selection,125,34


,page_name,total_events
0,Home,122574
1,DeviceDetail,108508
2,DeviceComparisonHome,18181
3,DeviceComparison,9685


In [30]:
### Xem nguyên văn event_data của 2 event ứng viên cho "So sánh"
for candidate_event in ["select_device_for_comparison", "add_to_comparison"]:
    df_sample_raw = query_cached(
        f"SELECT event_data FROM laplaptech.user_event_tracking "
        f"WHERE event_name = '{candidate_event}' LIMIT 5",
        cache_dir="../data/raw", force_refresh=False
    )
    print(f"--- {candidate_event} ---")
    for val in df_sample_raw["event_data"]:
        print(val)
    print()

--- select_device_for_comparison ---
{"device_id": 101}
{"device_id": 84}
{"device_id": 55}
{"device_id": 47}
{"device_id": 40}

--- add_to_comparison ---
{"device_id": "32"}
{"device_id": "85"}
{"device_id": "107"}
{"device_id": "3"}
{"device_id": "52"}



In [31]:
### Xem cấu trúc int_comparison_chart_device
df_int_sample = query_cached(
    "SELECT * FROM laplaptech.int_comparison_chart_device LIMIT 10",
    cache_dir="../data/raw", force_refresh=False
)
display(df_int_sample.dtypes.to_frame("dtype"))
print(df_int_sample.columns.tolist())

,dtype
event_id,int64
session_id,object
user_psuedo_id,object
event_local_ts,int64
laptop_model_id,int64
sort_by,object
sort_direction,object
laptop_model_name,object
brand_name,object


['event_id', 'session_id', 'user_psuedo_id', 'event_local_ts', 'laptop_model_id', 'sort_by', 'sort_direction', 'laptop_model_name', 'brand_name']


In [32]:
### So khớp theo TẬP HỢP thiết bị trong cùng session — phép kiểm định quyết định
# (Đã thử so khớp theo COUNT trước đó và bị loại: int_comparison_chart_device là snapshot
#  toàn bộ danh sách máy tại mỗi lần thao tác SORT, không phải log từng lần select/add,
#  nên so theo count luôn lệch giả tạo ~97-98% bất kể event nào -> không dùng được.)

df_chart_qa = query_cached(
    "SELECT session_id, laptop_model_id FROM laplaptech.int_comparison_chart_device",
    cache_dir="../data/raw", force_refresh=False
)
df_select_raw = query_cached(
    "SELECT session_id, toInt64OrNull(JSONExtractString(event_data, 'device_id')) AS device_id "
    "FROM laplaptech.user_event_tracking WHERE event_name = 'select_device_for_comparison'",
    cache_dir="../data/raw", force_refresh=False
)
df_add_raw = query_cached(
    "SELECT session_id, toInt64OrNull(JSONExtractString(event_data, 'device_id')) AS device_id "
    "FROM laplaptech.user_event_tracking WHERE event_name = 'add_to_comparison'",
    cache_dir="../data/raw", force_refresh=False
)

def containment_ratio(df_chart: pd.DataFrame, df_candidate: pd.DataFrame, label: str) -> float:
    """Tỷ lệ trung bình (theo session) số laptop_model_id trong chart nằm trong tập candidate cùng session."""
    chart_sets = df_chart.groupby("session_id")["laptop_model_id"].apply(set)
    cand_sets = df_candidate.groupby("session_id")["device_id"].apply(set)
    ratios = [
        len(chart_set & cand_sets.get(sid, set())) / len(chart_set)
        for sid, chart_set in chart_sets.items() if len(chart_set) > 0
    ]
    avg_ratio = sum(ratios) / len(ratios)
    print(f"{label}: containment trung bình = {avg_ratio:.1%} (trên {len(ratios)} session)")
    return avg_ratio

r_select = containment_ratio(df_chart_qa, df_select_raw, "select_device_for_comparison")
r_add = containment_ratio(df_chart_qa, df_add_raw, "add_to_comparison")

select_device_for_comparison: containment trung bình = 78.3% (trên 1154 session)
add_to_comparison: containment trung bình = 27.4% (trên 1154 session)


In [33]:
### QUYẾT ĐỊNH CUỐI CÙNG: công thức interest_score'
final_decision = f"""
## [Giai đoạn 3 - 0.4] Quyết định cuối cùng — interest_score'

Containment ratio: select_device_for_comparison = {r_select:.1%}, add_to_comparison = {r_add:.1%}
-> select_device_for_comparison đại diện đúng cho hành động đưa máy vào bảng so sánh.

**Công thức CHÍNH THỨC (thay thế interest_score gốc ở Giai đoạn 1):**

    interest_score' = 1 × (số pageview DeviceDetail) + 2 × (số lần select_device_for_comparison)

Thay đổi so với gốc: (1) bỏ thành phần video review — không có dữ liệu tracking;
(2) dùng select_device_for_comparison, KHÔNG dùng add_to_comparison cho thành phần "So sánh".

**Giới hạn:** containment {r_select:.1%} nghĩa là ~{100 - r_select*100:.1f}% thiết bị trong bảng
so sánh không khớp lần select gần nhất cùng session (có thể do độ trễ, do người dùng xóa bớt máy
trước khi sort). interest_score' là ước lượng, không phải phép đo tuyệt đối chính xác.
add_to_comparison (6.224 sự kiện) chưa rõ là hành động gì, không dùng ở đây, ghi vào backlog.
"""
append_note(final_decision)


In [34]:
### Build interest_score' và merge vào population sạch
interest_query = """
WITH pageview_cnt AS (
    SELECT toInt64OrNull(JSONExtractString(event_data, 'device_id')) AS laptop_id,
           COUNT(*) AS pageview_count
    FROM laplaptech.user_event_tracking
    WHERE event_name = 'pageview' AND JSONExtractString(event_data, 'page_name') = 'DeviceDetail'
    GROUP BY laptop_id
),
compare_cnt AS (
    SELECT toInt64OrNull(JSONExtractString(event_data, 'device_id')) AS laptop_id,
           COUNT(*) AS compare_count
    FROM laplaptech.user_event_tracking
    WHERE event_name = 'select_device_for_comparison'
    GROUP BY laptop_id
)
SELECT
    m.id AS laptop_model_id,
    COALESCE(p.pageview_count, 0) AS pageview_count,
    COALESCE(c.compare_count, 0)  AS compare_count,
    COALESCE(p.pageview_count, 0) * 1 + COALESCE(c.compare_count, 0) * 2 AS interest_score
FROM laplaptech.laptop_model m
LEFT JOIN pageview_cnt p ON p.laptop_id = m.id
LEFT JOIN compare_cnt  c ON c.laptop_id = m.id
"""  # Đồng bộ với sql/metrics.sql
df_interest = query_cached(interest_query, cache_dir="../data/raw", force_refresh=False)
print(f"Số laptop có interest_score > 0: {(df_interest['interest_score'] > 0).sum()}/{len(df_interest)}")
display(df_interest.sort_values("interest_score", ascending=False).head(10))

df_laptop_metrics = pd.read_csv("../data/processed/laptop_metrics.csv")

df_analysis_base = df_laptop_metrics.merge(
    df_interest[["laptop_model_id", "pageview_count", "compare_count", "interest_score"]],
    left_on="id", right_on="laptop_model_id", how="left"
)
df_analysis_base["interest_score"] = df_analysis_base["interest_score"].fillna(0)
df_analysis_base = select_report_columns(df_analysis_base)  # loại laptop_model_id_x/_y và cột thô còn sót
df_analysis_base.to_csv("../data/processed/analysis_base.csv", index=False)

print(f"analysis_base.csv: {len(df_analysis_base)} laptop (đã loại contamination, có interest_score')")

Số laptop có interest_score > 0: 154/161


,laptop_model_id,pageview_count,compare_count,interest_score
77,3,4223,1157,6537
26,85,4604,859,6322
34,32,2024,1096,4216
119,52,2516,791,4098
141,12,2170,872,3914
127,119,3087,374,3835
99,20,1318,878,3074
96,72,1659,704,3067
41,117,2473,223,2919
49,48,1790,508,2806


analysis_base.csv: 133 laptop (đã loại contamination, có interest_score')
